In [1]:
# Look at the spread of liquidation dates across our failed companies.
import sys
sys.path.append("..")
import json
from pathlib import Path
import pandas as pd

cohort = pd.read_parquet("../data/cohort.parquet")
failed = cohort[cohort["is_failed"] == 1]

dates = []
for number in failed["CompanyNumber"]:
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    dates.append(data["profile"].get("date_of_cessation"))

pd.Series(dates).notna().sum()

np.int64(78)

In [2]:
# Look at the filing history of one failed company to find where liquidation is recorded.
example = failed["CompanyNumber"].iloc[0]
data = json.loads((Path("../data/raw") / f"{example}.json").read_text())

for item in data["filings"]["items"]:
    print(item.get("date"), "-", item.get("type"), "-", item.get("description", "")[:80])

2026-01-20 - LIQ03 - liquidation-voluntary-statement-of-receipts-and-payments-with-brought-down-date
2025-04-25 - AD01 - change-registered-office-address-company-with-date-old-address-new-address
2024-12-04 - NDISC - liquidation-disclaimer-notice
2024-12-04 - NDISC - liquidation-disclaimer-notice
2024-12-04 - NDISC - liquidation-disclaimer-notice
2024-12-03 - AD01 - change-registered-office-address-company-with-date-old-address-new-address
2024-11-26 - LIQ02 - liquidation-voluntary-statement-of-affairs
2024-11-26 - 600 - liquidation-voluntary-appointment-of-liquidator
2024-11-26 - RESOLUTIONS - resolution
2024-05-29 - DISS40 - gazette-filings-brought-up-to-date
2024-05-28 - GAZ1 - gazette-notice-compulsory
2024-05-24 - AA - accounts-with-accounts-type-micro-entity
2024-02-13 - MR04 - mortgage-satisfy-charge-full
2023-10-24 - CS01 - confirmation-statement-with-updates
2023-09-11 - AA01 - change-account-reference-date-company-previous-extended
2022-11-30 - CS01 - confirmation-statement-w

In [3]:
# Check what filing types most commonly mark the start of liquidation, across all failed companies.
from collections import Counter

liquidation_types = Counter()
for number in failed["CompanyNumber"]:
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    for item in data["filings"]["items"]:
        t = item.get("type", "")
        if "LIQ" in t or t in ("600", "GAZ1", "RESOLUTIONS"):
            liquidation_types[t] += 1

liquidation_types.most_common(20)

[('LIQ03', 1209),
 ('RESOLUTIONS', 1122),
 ('600', 1029),
 ('GAZ1', 731),
 ('LIQ02', 643),
 ('LIQ01', 174),
 ('LIQ10', 119),
 ('LIQ14', 94),
 ('LIQ13', 40),
 ('LIQ06', 25),
 ('LIQ MISC', 21),
 ('LIQ MISC OC', 13),
 ('LIQ', 12),
 ('COLIQ', 6),
 ('LIQ07', 3),
 ('LIQ04', 2),
 ('LIQ09', 1),
 ('LIQ MISC RES', 1)]

In [4]:
# Check how many failed companies have a '600' filing vs none at all.
has_600 = 0
missing_600 = []
for number in failed["CompanyNumber"]:
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    types = [item.get("type") for item in data["filings"]["items"]]
    if "600" in types:
        has_600 += 1
    else:
        missing_600.append(number)

print(has_600, len(missing_600))

899 601


In [5]:
# Look at the filing history of a company missing the '600' filing, to find its equivalent marker.
example2 = missing_600[0]
data2 = json.loads((Path("../data/raw") / f"{example2}.json").read_text())

for item in data2["filings"]["items"]:
    print(item.get("date"), "-", item.get("type"), "-", item.get("description", "")[:80])

2025-10-03 - WU07 - liquidation-compulsory-winding-up-progress-report
2024-10-05 - WU07 - liquidation-compulsory-winding-up-progress-report
2024-06-22 - CVA3 - liquidation-cva-supervisors-abstract-of-receipts-payments-with-brought-down-date
2023-09-02 - WU04 - liquidation-compulsory-appointment-liquidator
2023-09-01 - COCOMP - liquidation-compulsory-winding-up-order
2023-08-22 - AD01 - change-registered-office-address-company-with-date-old-address-new-address
2023-06-12 - CVA3 - liquidation-cva-supervisors-abstract-of-receipts-payments-with-brought-down-date
2022-07-28 - CS01 - confirmation-statement-with-no-updates
2022-06-08 - CVA3 - liquidation-cva-supervisors-abstract-of-receipts-payments-with-brought-down-date
2021-08-02 - AA - accounts-with-accounts-type-micro-entity
2021-07-13 - MR04 - mortgage-satisfy-charge-full
2021-06-08 - CVA3 - liquidation-cva-supervisors-abstract-of-receipts-payments-with-brought-down-date
2021-04-29 - CS01 - confirmation-statement-with-no-updates
2020-09

In [6]:
# Check how many of the 601 companies missing '600' have a COCOMP filing instead.
has_cocomp = 0
still_missing = []
for number in missing_600:
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    types = [item.get("type") for item in data["filings"]["items"]]
    if "COCOMP" in types:
        has_cocomp += 1
    else:
        still_missing.append(number)

print(has_cocomp, len(still_missing))

354 247


In [7]:
# Looking at what's happening in a company still missing both '600' and 'COCOMP'.
example3 = still_missing[0]
data3 = json.loads((Path("../data/raw") / f"{example3}.json").read_text())

for item in data3["filings"]["items"]:
    print(item.get("date"), "-", item.get("type"), "-", item.get("description", "")[:80])

2009-09-16 - 405(1) - legacy
2009-07-03 - AA - accounts-with-accounts-type-total-exemption-small
2009-02-24 - 363a - legacy
2007-11-26 - 363s - legacy
2007-06-25 - AA - accounts-with-accounts-type-total-exemption-small
2007-04-10 - 287 - legacy
2007-04-10 - 363s - legacy
2006-08-29 - AA - accounts-with-accounts-type-total-exemption-small
2006-08-18 - 288b - legacy
2005-12-23 - 363s - legacy
2005-09-24 - 395 - legacy
2005-04-11 - 225 - legacy
2005-04-05 - 88(2)R - legacy
2005-04-05 - 288a - legacy
2005-04-05 - 288a - legacy
2005-04-05 - 288a - legacy
2004-12-22 - 288a - legacy
2004-10-27 - 288a - legacy
2004-10-27 - 288a - legacy
2004-10-21 - 288b - legacy
2004-10-21 - 288b - legacy
2004-10-21 - NEWINC - incorporation-company


In [8]:
# Check the most recent filing date for each of the 247 unresolved companies.
recent_dates = []
for number in still_missing:
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    items = data["filings"]["items"]
    if items:
        recent_dates.append(max(item["date"] for item in items))
    else:
        recent_dates.append(None)

pd.Series(recent_dates).describe()

count            246
unique           186
top       2018-07-10
freq              10
dtype: object

In [9]:
still_missing_status = cohort[cohort["CompanyNumber"].isin(still_missing)]["CompanyStatus"].value_counts()
still_missing_status

CompanyStatus
Liquidation                                         149
In Administration                                    48
Live but Receiver Manager on at least one charge     33
Voluntary Arrangement                                 7
In Administration/Administrative Receiver             4
ADMINISTRATIVE RECEIVER                               3
ADMINISTRATION ORDER                                  2
In Administration/Receiver Manager                    1
Name: count, dtype: int64

In [10]:
# Look at filing types for a Liquidation-status company still missing 600/COCOMP.
liq_unresolved = still_missing_status.index  # not quite right, need the actual company numbers
liq_unresolved_numbers = cohort[
    (cohort["CompanyNumber"].isin(still_missing)) & (cohort["CompanyStatus"] == "Liquidation")
]["CompanyNumber"]

example4 = liq_unresolved_numbers.iloc[0]
data4 = json.loads((Path("../data/raw") / f"{example4}.json").read_text())
for item in data4["filings"]["items"]:
    print(item.get("date"), "-", item.get("type"), "-", item.get("description", "")[:80])

1988-01-08 - SC71 - legacy
1987-01-01 - PRE87 - selection-of-documents-registered-before-January-1987
1986-05-29 - SC71 - legacy
1986-05-23 - SC71 - legacy


In [11]:
# Find the failure date for a company using the 600 or COCOMP filing, whichever is present.
def get_failure_date(filings_items):
    """Return the date liquidation started, or None if no clear marker exists."""
    for target_type in ("600", "COCOMP"):
        matches = [item["date"] for item in filings_items if item.get("type") == target_type]
        if matches:
            return min(matches)
    return None

In [12]:
# Apply the failure date rule across all failed companies and check coverage.
failure_dates = {}
for number in failed["CompanyNumber"]:
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    date = get_failure_date(data["filings"]["items"])
    if date:
        failure_dates[number] = date

print(len(failure_dates))

1253
